In [ ]:
from diffusers import StableDiffusionPipeline
import torch
import matplotlib.pyplot as plt

# LoRA (Low-Rank Adaptation) is a fine-tuning technique that injects small, trainable rank-decomposition matrices into each layer of the Transformer architecture.
# Unlike full fine-tuning, LoRA allows you to drastically change the model's artistic style or add specific characters while keeping the main model weights frozen and the file size very small (megabytes instead of gigabytes).

# select model
model_id = "runwayml/stable-diffusion-v1-5"

# load pipeline
pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float32).to("cpu")

#Load LoRA Weights "Paper_Cutting"
pipe.load_lora_weights("Kontext-Style/Paper_Cutting_lora", weight_name="Paper_Cutting_lora_weights.safetensors")

# use CPU
device = "cpu"
pipe = pipe.to(device)

# our prompt for CLIP
# we add "papercut style" to the prompt to trigger the LoRA features
prompt = "a cute fluffy cat, Paper_Cutting style, vibrant colors, soft studio lighting, high detail"


# function-callback which call in each step of scheduler
def latents_callback(step, timestamp, latents):
  print(f"Step {step}. Current value of noise: {latents.mean().item():.4f}")

# generation
print("starting of generation...")


result_image = pipe(
    prompt=prompt,
    num_inference_steps=20, # num_inference_steps - scheduler for U-Net
    guidance_scale=7.5,
    callback=latents_callback,
    callback_steps=1 # call 'latents_callback' each step
).images[0]

result_image.save("lora_papercut_cat.png")

plt.imshow(result_image)
plt.axis("off")
plt.show()

print("Finished!")
